In [9]:
# import external libraries
import joblib
import os
import pandas as pd
import numpy as np
import warnings
import logging
from typing import Tuple


# import sklearn libraries
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier,plot_tree
from sklearn.metrics import accuracy_score, log_loss, accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import ShuffleSplit, cross_validate
from sklearn.ensemble import BaggingClassifier, GradientBoostingClassifier
from sklearn.feature_selection import RFE

# set global constants
GLOBAL = {
    'DATA_PATH':'data/Loan_default.csv',
    'TARGET_VARIABLE':'Default',
    'RANDOM_STATE':10,
    'TEST_SIZE':0.2
}

# ignore warnings
warnings.filterwarnings('ignore')

# set logging configuration
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)


In [10]:

# function to load the dataset
def load_data(file_path: str) -> pd.DataFrame:
    """Loads raw dataset from a CSV file."""
    if not os.path.exists(file_path):
        logger.error(f"File not found at: {file_path}")
        raise FileNotFoundError(f"Missing input data: {file_path}")

    logger.info(f"Loading data from {file_path}")

    df = pd.read_csv(file_path)
    logger.info(f"Data shape: {df.shape}")
    return df

try:
    # call the function
    df = load_data(GLOBAL['DATA_PATH'])

    # create a copy of the df 
    df_copy = pd.DataFrame.copy(df)

except FileNotFoundError:
    print(f"Missing input data")



2026-07-22 22:11:34,569 - INFO - Loading data from data/Loan_default.csv
2026-07-22 22:11:35,074 - INFO - Data shape: (255347, 18)


In [11]:
# ===========================
# 1. Exploratory Data Analysis
# ===========================

def inspect_dataframe(df: pd.DataFrame) -> None:
    """Features of the dataset"""
    
    # print the column names
    print("---Columns---")
    print(df_copy.columns)

    # print the shape of the data frame
    print("---Shape of df---")
    print(df_copy.shape)

    # print the datatypes
    print("---Datatypes of columns---")
    print(df_copy.dtypes)

    # print the unique values
    print("Unique Values of columns")
    for x in df_copy.columns:
        print(df_copy[x].unique())

    # print the missing values
    print("--- Missing Values ---")
    print(df_copy.isnull().sum()[df.isnull().sum() > 0])

    # print the statistical values
    print("\n--- Summary Statistics ---")
    print(df_copy.describe())

    # print the details about the columns
    print("\n---Details of the columns---")
    print(df_copy.info())

    # drop unwanted columns
    print("\n---Drop unwanted Column---")
    print(df_copy.drop(columns=['LoanID'],inplace=True))


inspect_dataframe(df_copy)


---Columns---
Index(['LoanID', 'Age', 'Income', 'LoanAmount', 'CreditScore',
       'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'LoanTerm',
       'DTIRatio', 'Education', 'EmploymentType', 'MaritalStatus',
       'HasMortgage', 'HasDependents', 'LoanPurpose', 'HasCoSigner',
       'Default'],
      dtype='object')
---Shape of df---
(255347, 18)
---Datatypes of columns---
LoanID             object
Age                 int64
Income              int64
LoanAmount          int64
CreditScore         int64
MonthsEmployed      int64
NumCreditLines      int64
InterestRate      float64
LoanTerm            int64
DTIRatio          float64
Education          object
EmploymentType     object
MaritalStatus      object
HasMortgage        object
HasDependents      object
LoanPurpose        object
HasCoSigner        object
Default             int64
dtype: object
Unique Values of columns
['I38PQUQS96' 'HPSK72WA7R' 'C1OZ6DPJ8Y' ... 'XQK1UUUNGP' 'JAO28CPL4H'
 'ZTH91CGL0B']
[56 69 46 32 60 25 38 36 

In [12]:

# ===================================
# 3. Data Preprocessing
# ===================================

def data_preprocessing() -> Tuple[np.array, np.array]:
    """PREPROCESS THE DATA BY REMOVING NULL VALUES, ENCODING CATEGORICAL COLUMNS AND SCALING THE DATA."""

    # missing values
    print("---Check for Missing Values---")
    columns_with_missing_values = []
    for col in df_copy.columns:
        if df_copy[col].isnull().sum()>0:
            columns_with_missing_values.append(col)
    if len(columns_with_missing_values)>0:
        print("Missing Values: ")
        for col in df_copy[columns_with_missing_values].columns:
            print(df_copy[col],": ", df_copy[col].isnull().sum())
    else:
        print("No missing values found in the dataset.")

    # datatypes of columns
    print("---Datatype of columns---")
    categorical_cols = []
    for col in df_copy.columns:
        if df_copy[col].dtype=='object':
            categorical_cols.append(col)
    
    if len(categorical_cols)>0:
        print("Categorical Columns : ",categorical_cols)

        encoder = {}
        le = LabelEncoder()
        for col in categorical_cols:
            df_copy[col] = le.fit_transform(df_copy[col])
            encoder[col] = le
    else:
        print("No categorical variables found. Encoding Not needed.")

    # scaling of features
    print("---Feature Scaling---")
    X = df_copy.values[:,:-1]
    Y = df_copy.values[:,-1]

    scaler = StandardScaler()
    scaler.fit(X)
    X = scaler.transform(X)
    print("Scaled Features : \n",X)
    return X,Y, scaler
# call the function
X,Y, scaler = data_preprocessing()

    

---Check for Missing Values---
No missing values found in the dataset.
---Datatype of columns---
Categorical Columns :  ['Education', 'EmploymentType', 'MaritalStatus', 'HasMortgage', 'HasDependents', 'LoanPurpose', 'HasCoSigner']
---Feature Scaling---
Scaled Features : 
 [[ 8.33989509e-01  8.96928115e-02 -1.08683299e+00 ...  9.99463619e-01
   1.41535378e+00  9.99784630e-01]
 [ 1.70122109e+00 -8.23020714e-01 -4.43088703e-02 ... -1.00053667e+00
   1.41535378e+00  9.99784630e-01]
 [ 1.66888295e-01  4.38543784e-02  2.27148729e-02 ...  9.99463619e-01
  -1.41606345e+00 -1.00021542e+00]
 ...
 [ 8.33989509e-01  5.95616130e-02  1.13939142e+00 ...  9.99463619e-01
  -1.41606345e+00  9.99784630e-01]
 [-9.99521902e-02  6.69789183e-02 -9.45840328e-01 ...  9.99463619e-01
   1.41535378e+00 -1.00021542e+00]
 [ 1.23425024e+00 -1.54201168e+00 -1.54004788e+00 ... -1.00053667e+00
  -3.54832254e-04  9.99784630e-01]]


In [13]:
# =============================
# 4. Random Forest Implementation
# =============================
def train_test_data(X: np.array,Y:np.array,GLOBAL) -> Tuple[np.array,np.array]:
    """Train and Test the Decision Tree Classifier."""

    # split data into training and testing data
    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=GLOBAL['TEST_SIZE'], random_state=GLOBAL['RANDOM_STATE'])

    # initialize the decision tree
    DT = DecisionTreeClassifier(criterion='gini')

    # Recursive Feature Elimination
    rfe_DT = RFE(estimator=DT, n_features_to_select=10)

    # train the decision tree
    rfe_DT.fit(X_train,Y_train)

    # test the decision tree
    Y_pred_train = rfe_DT.predict(X_train)
    Y_pred_test = rfe_DT.predict(X_test)
    Y_pred_train_proba = rfe_DT.predict_proba(X_train)
    Y_pred_test_proba = rfe_DT.predict_proba(X_test)

    # evaluation of accuracy
    training_accuracy  = accuracy_score(Y_train,Y_pred_train)
    test_accuracy  = accuracy_score(Y_test,Y_pred_test)
    training_loss  = log_loss(Y_train,Y_pred_train_proba)
    test_loss  = log_loss(Y_test,Y_pred_test_proba)

    precision = precision_score(Y_train,Y_pred_train,average='macro')
    recall = recall_score(Y_train,Y_pred_train,average='macro')
    f1score = f1_score(Y_train,Y_pred_train,average='macro')

    # print the result
    print("training accuracy:       ", np.round(training_accuracy,2))
    print("test accuracy:           ", np.round(test_accuracy,2))
    print("trianing loss:           ", np.round(training_loss,2))
    print("test loss:               ", np.round(test_loss,2))
    print("Precision Score:         ",np.round(precision,2))
    print("Recall:                  ",np.round(recall,2))
    print("F1 Score:                ",np.round(f1score,2))

    # # Set up ShuffleSplit cross-validator
    # shuffle_split = ShuffleSplit(n_splits=100, test_size=GLOBAL['TEST_SIZE'], random_state=GLOBAL['RANDOM_STATE'])

    # # Perform cross-validation and collect both train and test scores
    # cv_results = cross_validate(rfe_DT, X, Y, cv=shuffle_split, scoring='accuracy', return_train_score=True)

    # # Extract train and test scores
    # train_scores = cv_results['train_score']
    # test_scores = cv_results['test_score']

    # # Show individual scores and their means
    # print("cross validation: train accuracy:" , np.round(train_scores.mean(),2))
    # print("cross validation: test accuracy:" , np.round(test_scores.mean(),2))

    return X_train,X_test,Y_train,Y_test
# call the function
X_train,X_test,Y_train,Y_test = train_test_data(X,Y,GLOBAL)


training accuracy:        1.0
test accuracy:            0.8
trianing loss:            0.0
test loss:                7.13
Precision Score:          1.0
Recall:                   1.0
F1 Score:                 1.0


In [14]:

# BAGGING CLASSIFIER

def bagging_classifier(X_train,Y_train,Y_test):

    # Bagging Classifier
    bagging_model = BaggingClassifier(estimator=DecisionTreeClassifier(criterion='gini'),
                                  n_estimators=100, random_state=42,
                                  max_samples=0.6,max_features=0.7)
    # train the bagging model
    bagging_model.fit(X_train, Y_train)

    # test the bagging model
    y_pred_train_bagging = bagging_model.predict(X_train)
    y_pred_test_bagging = bagging_model.predict(X_test)

    # predict the probabilities
    y_prob_train_bagging = bagging_model.predict_proba(X_train)
    y_prob_test_bagging = bagging_model.predict_proba(X_test)

    # get accuracy score
    training_accuracy_bagging = accuracy_score(Y_train, y_pred_train_bagging)
    test_accuracy_bagging = accuracy_score(Y_test, y_pred_test_bagging)

    # loss in training and testing
    training_loss_bagging = log_loss(Y_train, y_prob_train_bagging)
    test_loss_bagging = log_loss(Y_test, y_prob_test_bagging)

    # print the result
    print("Bagging - Training Accuracy:     ", np.round(training_accuracy_bagging,2))
    print("Bagging - Test Accuracy:         ", np.round(test_accuracy_bagging,2))
    print("Bagging - Training Loss:         ", np.round(training_loss_bagging,2))
    print("Bagging - Test Loss:             ", np.round(test_loss_bagging,2))

# call the method
bagging_classifier(X_train,Y_train,Y_test)


Bagging - Training Accuracy:      0.97
Bagging - Test Accuracy:          0.88
Bagging - Training Loss:          0.13
Bagging - Test Loss:              0.33


In [15]:

def boosting_classifier(X_train, Y_train, Y_test):

    # Gradient Boosting Classifier
    gb_model = GradientBoostingClassifier(n_estimators=100, random_state=42,learning_rate=0.1)
    gb_model.fit(X_train, Y_train)

    # train the model
    y_pred_train_gb = gb_model.predict(X_train)
    y_pred_test_gb = gb_model.predict(X_test)

    # get the probabilities
    y_prob_train_gb = gb_model.predict_proba(X_train)
    y_prob_test_gb = gb_model.predict_proba(X_test)

    # get the accuracy score and loss values
    training_accuracy_gb = accuracy_score(Y_train, y_pred_train_gb)
    test_accuracy_gb = accuracy_score(Y_test, y_pred_test_gb)
    training_loss_gb = log_loss(Y_train, y_prob_train_gb)
    test_loss_gb = log_loss(Y_test, y_prob_test_gb)

    # print all the result
    print("Gradient Boosting - Training Accuracy:", np.round(training_accuracy_gb, 2))
    print("Gradient Boosting - Test Accuracy:", np.round(test_accuracy_gb, 2))
    print("Gradient Boosting - Training Loss:", np.round(training_loss_gb, 2))
    print("Gradient Boosting - Test Loss:", np.round(test_loss_gb, 2))
    return gb_model
# call the function
gb_model = boosting_classifier(X_train,Y_train,Y_test)

Gradient Boosting - Training Accuracy: 0.89
Gradient Boosting - Test Accuracy: 0.89
Gradient Boosting - Training Loss: 0.31
Gradient Boosting - Test Loss: 0.31


In [17]:

import joblib
joblib.dump(gb_model,'fastapi/model.pkl')
joblib.dump(scaler,'fastapi/scaler.pkl')
joblib.dump(X,'fastapi/X.pkl')
joblib.dump(Y,'fastapi/Y.pkl')


['fastapi/Y.pkl']